# BSXScript 3.1 二進位檔案分析教學

## 使用 Python struct 庫解析 bsxx.dat 遊戲腳本檔案

本教學將帶您深入了解如何分析 Bishop 引擎視覺小說遊戲的腳本檔案 `bsxx.dat`。

### 學習目標

1. 了解 BSXScript 3.1 檔案的整體結構
2. 學會使用 Python `struct` 模組解析二進位資料
3. 定位並解析各個資料區域：
   - 代碼區 (Code Section)
   - 名稱索引表 (Name Index Table)
   - 名稱字串表 (Name String Table)
   - 對話索引表 (Dialog Index Table)
   - 對話字串表 (Dialog String Table)
4. 提取遊戲中的角色名稱和對話文本

### BSXScript 3.1 檔案結構概覽

```
┌─────────────────────────────────────────────────────────┐
│ 0x00000 - 0x00010: Magic "BSXScript 3.1"               │
├─────────────────────────────────────────────────────────┤
│ 0x00010 - 0x00030: Header (版本、計數器等)              │
├─────────────────────────────────────────────────────────┤
│ 0x00030 - 0x000B0: Section Table                       │
├─────────────────────────────────────────────────────────┤
│ ...代碼區 (指令、跳轉表等)...                          │
├─────────────────────────────────────────────────────────┤
│ 0x68BC0 - 0x68C14: 名稱索引表 (21 × 4 bytes)           │
├─────────────────────────────────────────────────────────┤
│ 0x68C14 - 0x68C20: 填充區                              │
├─────────────────────────────────────────────────────────┤
│ 0x68C20 - 0x68D0E: 名稱字串表 (UTF-16LE)               │
├─────────────────────────────────────────────────────────┤
│ 0x68D10 - 0x6A210: 對話索引表 (1344 × 4 bytes)         │
├─────────────────────────────────────────────────────────┤
│ 0x6A210 - EOF:     對話字串表 (UTF-16LE)               │
└─────────────────────────────────────────────────────────┘
```

## 1. 匯入必要的函式庫

首先，我們需要匯入 Python 的 `struct` 模組和其他必要的函式庫。

### struct 模組簡介

`struct` 模組提供了將 Python 值與 C 結構體之間進行轉換的功能。常用的格式字元包括：

| 格式 | C 類型 | Python 類型 | 大小 (bytes) |
|------|--------|-------------|--------------|
| `<`  | 小端序 (little-endian) | - | - |
| `>`  | 大端序 (big-endian) | - | - |
| `B`  | unsigned char | int | 1 |
| `H`  | unsigned short | int | 2 |
| `I`  | unsigned int | int | 4 |
| `s`  | char[] | bytes | - |

例如：`<I` 表示以小端序讀取一個 4 位元組的無符號整數。

In [1]:
import struct
import os
from pathlib import Path

# 設定檔案路徑
SCRIPT_FILE = Path("../BSA/bsxx.dat.bak")  # 使用原始備份檔案

# 確認檔案存在
if SCRIPT_FILE.exists():
    file_size = os.path.getsize(SCRIPT_FILE)
    print(f"✓ 檔案存在: {SCRIPT_FILE}")
    print(f"✓ 檔案大小: {file_size:,} bytes ({file_size / 1024:.2f} KB)")
else:
    print(f"✗ 找不到檔案: {SCRIPT_FILE}")

✓ 檔案存在: ../BSA/bsxx.dat.bak
✓ 檔案大小: 546,842 bytes (534.03 KB)


## 2. 讀取二進位檔案

使用 Python 的 `open()` 函式以二進位模式 (`'rb'`) 讀取整個檔案到記憶體中。

**重要提示**：對於大型檔案，您可能需要考慮使用記憶體映射 (`mmap`) 或分段讀取。但對於 500KB 左右的腳本檔案，直接讀入記憶體是安全的。

In [2]:
# 以二進位模式讀取檔案
with open(SCRIPT_FILE, 'rb') as f:
    data = f.read()

print(f"讀取了 {len(data):,} bytes 到記憶體")
print(f"\n前 64 bytes (十六進位):")
print(' '.join(f'{b:02X}' for b in data[:64]))

讀取了 546,842 bytes 到記憶體

前 64 bytes (十六進位):
42 53 58 53 63 72 69 70 74 20 33 2E 31 00 00 00 00 04 00 00 02 00 00 00 00 00 00 00 01 00 00 00 24 00 00 00 00 00 00 00 07 00 00 00 E0 00 00 00 1B 18 04 00 3F 05 00 00 00 19 04 00 60 33 00 00


## 3. 解析檔案標頭結構

BSXScript 3.1 檔案的標頭包含以下關鍵資訊：

| 偏移量 | 大小 | 內容 |
|--------|------|------|
| 0x00 | 16 bytes | Magic: "BSXScript 3.1\x00\x00\x00" |
| 0x10 | 4 bytes | Version (0x400) |
| 0x14 | 4 bytes | Entry count |
| 0x18 | 4 bytes | Section count |
| 0x90 | 4 bytes | 名稱字串表偏移量 |
| 0x98 | 4 bytes | 對話索引表偏移量 |
| 0xA0 | 4 bytes | 對話字串表偏移量 |

### 使用 struct.unpack_from() 解析

`struct.unpack_from(format, buffer, offset)` 函式可以從指定偏移量開始解析資料：
- `format`: 格式字串
- `buffer`: 二進位資料
- `offset`: 起始偏移量

In [6]:
# 解析 Magic Number (前 16 bytes)
magic = data[0:16]
print("【Magic Number】")
print(f"  原始 bytes: {magic}")
decoded_text = magic.rstrip(b'\x00').decode('ascii')
print(f"  解碼文字: {decoded_text}")

# 解析版本和計數器資訊
# struct.unpack_from('<I', data, offset) 從 offset 位置讀取一個小端序 4-byte 整數
version = struct.unpack_from('<I', data, 0x10)[0]
entry_count = struct.unpack_from('<I', data, 0x14)[0]
section_count = struct.unpack_from('<I', data, 0x18)[0]

print("\n【Header 資訊】")
print(f"  Version: 0x{version:X}")
print(f"  Entry count: {entry_count}")
print(f"  Section count: {section_count}")

【Magic Number】
  原始 bytes: b'BSXScript 3.1\x00\x00\x00'
  解碼文字: BSXScript 3.1

【Header 資訊】
  Version: 0x400
  Entry count: 2
  Section count: 0


In [7]:
# 解析 Header 中的關鍵偏移量
# 這些偏移量告訴我們各個資料區域的位置

name_string_table_offset = struct.unpack_from('<I', data, 0x90)[0]
dialog_index_table_offset = struct.unpack_from('<I', data, 0x98)[0]
dialog_string_table_offset = struct.unpack_from('<I', data, 0xA0)[0]

print("【Header 中的偏移量指標】")
print(f"  0x90 → 名稱字串表偏移量: 0x{name_string_table_offset:05X}")
print(f"  0x98 → 對話索引表偏移量: 0x{dialog_index_table_offset:05X}")
print(f"  0xA0 → 對話字串表偏移量: 0x{dialog_string_table_offset:05X}")

print("\n【區域大小計算】")
name_string_size = dialog_index_table_offset - name_string_table_offset
dialog_index_size = dialog_string_table_offset - dialog_index_table_offset
dialog_string_size = len(data) - dialog_string_table_offset

print(f"  名稱字串表大小: {name_string_size} bytes (0x{name_string_size:X})")
print(f"  對話索引表大小: {dialog_index_size} bytes ({dialog_index_size // 4} 個索引)")
print(f"  對話字串表大小: {dialog_string_size} bytes")

【Header 中的偏移量指標】
  0x90 → 名稱字串表偏移量: 0x68C20
  0x98 → 對話索引表偏移量: 0x68D10
  0xA0 → 對話字串表偏移量: 0x6A210

【區域大小計算】
  名稱字串表大小: 240 bytes (0xF0)
  對話索引表大小: 5376 bytes (1344 個索引)
  對話字串表大小: 112138 bytes


## 4. 定位代碼區 (Code Section)

代碼區包含遊戲腳本的執行指令，從檔案開頭一直延伸到名稱索引表開始的位置 (0x68BC0)。

```
代碼區範圍: 0x00000 - 0x68BC0 (約 428 KB)
```

這個區域包含：
- 腳本指令 (跳轉、條件判斷等)
- 場景定義
- 事件觸發器
- 其他遊戲邏輯

**注意**：在修改翻譯時，代碼區必須保持不變，否則會破壞遊戲邏輯。

In [8]:
# 名稱索引表的起始位置
NAME_INDEX_TABLE_OFFSET = 0x68BC0

# 代碼區範圍
code_section_start = 0x00
code_section_end = NAME_INDEX_TABLE_OFFSET
code_section_size = code_section_end - code_section_start

print("【代碼區定位】")
print(f"  起始位置: 0x{code_section_start:05X}")
print(f"  結束位置: 0x{code_section_end:05X}")
print(f"  區域大小: {code_section_size:,} bytes ({code_section_size / 1024:.2f} KB)")

# 顯示代碼區開頭的內容
print("\n【代碼區前 32 bytes】")
code_start = data[0:32]
print(' '.join(f'{b:02X}' for b in code_start))

# 顯示代碼區結尾的內容 (接近名稱索引表的位置)
print(f"\n【代碼區結尾 (0x68BA0-0x68BC0)】")
code_end = data[0x68BA0:0x68BC0]
print(' '.join(f'{b:02X}' for b in code_end))

【代碼區定位】
  起始位置: 0x00000
  結束位置: 0x68BC0
  區域大小: 428,992 bytes (418.94 KB)

【代碼區前 32 bytes】
42 53 58 53 63 72 69 70 74 20 33 2E 31 00 00 00 00 04 00 00 02 00 00 00 00 00 00 00 01 00 00 00

【代碼區結尾 (0x68BA0-0x68BC0)】
6E 00 00 00 23 00 73 00 63 00 65 00 6E 00 65 00 5F 00 72 00 65 00 70 00 6C 00 61 00 79 00 00 00


## 5. 解析名稱索引表 (Name Index Table)

名稱索引表位於 `0x68BC0`，包含 21 個角色名稱的**字元偏移量**。

### 索引表結構

```
0x68BC0: [索引0] [索引1] [索引2] ... [索引20]
         (4 bytes each, little-endian 32-bit integer)
```

### 字元偏移量的意義

每個索引值是一個**字元偏移量**（不是位元組偏移量）。計算實際位置的公式：

```
實際位元組偏移量 = 名稱字串表起始 + (索引值 × 2)
                = 0x68C20 + (index × 2)
```

（乘以 2 是因為 UTF-16LE 每個字元佔 2 bytes）

In [9]:
# 名稱索引表和字串表的位置常數
NAME_INDEX_TABLE = 0x68BC0
NAME_STRING_TABLE = 0x68C20
NAME_COUNT = 21

print("【名稱索引表解析】")
print(f"  索引表位置: 0x{NAME_INDEX_TABLE:05X}")
print(f"  字串表位置: 0x{NAME_STRING_TABLE:05X}")
print(f"  名稱數量: {NAME_COUNT}")
print()

# 讀取並解析所有名稱索引
name_indices = []
for i in range(NAME_COUNT):
    # 使用 struct.unpack_from 從指定偏移量讀取 4-byte 整數
    offset = NAME_INDEX_TABLE + i * 4
    char_index = struct.unpack_from('<I', data, offset)[0]
    name_indices.append(char_index)
    print(f"  索引 [{i:2d}] 位置 0x{offset:05X}: 字元偏移量 = {char_index:3d}")

print(f"\n原始索引值: {name_indices}")

【名稱索引表解析】
  索引表位置: 0x68BC0
  字串表位置: 0x68C20
  名稱數量: 21

  索引 [ 0] 位置 0x68BC0: 字元偏移量 =   0
  索引 [ 1] 位置 0x68BC4: 字元偏移量 =   3
  索引 [ 2] 位置 0x68BC8: 字元偏移量 =   7
  索引 [ 3] 位置 0x68BCC: 字元偏移量 =  10
  索引 [ 4] 位置 0x68BD0: 字元偏移量 =  12
  索引 [ 5] 位置 0x68BD4: 字元偏移量 =  18
  索引 [ 6] 位置 0x68BD8: 字元偏移量 =  24
  索引 [ 7] 位置 0x68BDC: 字元偏移量 =  30
  索引 [ 8] 位置 0x68BE0: 字元偏移量 =  36
  索引 [ 9] 位置 0x68BE4: 字元偏移量 =  42
  索引 [10] 位置 0x68BE8: 字元偏移量 =  48
  索引 [11] 位置 0x68BEC: 字元偏移量 =  54
  索引 [12] 位置 0x68BF0: 字元偏移量 =  60
  索引 [13] 位置 0x68BF4: 字元偏移量 =  66
  索引 [14] 位置 0x68BF8: 字元偏移量 =  71
  索引 [15] 位置 0x68BFC: 字元偏移量 =  89
  索引 [16] 位置 0x68C00: 字元偏移量 =  95
  索引 [17] 位置 0x68C04: 字元偏移量 = 100
  索引 [18] 位置 0x68C08: 字元偏移量 = 103
  索引 [19] 位置 0x68C0C: 字元偏移量 = 106
  索引 [20] 位置 0x68C10: 字元偏移量 = 111

原始索引值: [0, 3, 7, 10, 12, 18, 24, 30, 36, 42, 48, 54, 60, 66, 71, 89, 95, 100, 103, 106, 111]


In [10]:
def read_utf16le_string(data: bytes, offset: int) -> str:
    """
    從指定偏移量讀取 UTF-16LE 編碼的字串（以 null 結尾）
    
    Args:
        data: 二進位資料
        offset: 起始偏移量
        
    Returns:
        解碼後的字串
    """
    end = offset
    # 尋找 null terminator (0x00 0x00)
    while end < len(data) - 1:
        if data[end] == 0 and data[end + 1] == 0:
            break
        end += 2  # UTF-16LE 每個字元 2 bytes
    
    # 解碼字串
    try:
        return data[offset:end].decode('utf-16le')
    except UnicodeDecodeError:
        return f"(解碼錯誤 at 0x{offset:X})"

# 使用索引表讀取所有角色名稱
print("【使用索引表讀取角色名稱】\n")

for i, char_index in enumerate(name_indices):
    # 計算實際位元組偏移量
    byte_offset = NAME_STRING_TABLE + char_index * 2
    
    # 讀取字串
    name = read_utf16le_string(data, byte_offset)
    
    print(f"  [{i:2d}] 字元偏移={char_index:3d} → 位元組偏移=0x{byte_offset:05X} → \"{name}\"")

【使用索引表讀取角色名稱】

  [ 0] 字元偏移=  0 → 位元組偏移=0x68C20 → "\B"
  [ 1] 字元偏移=  3 → 位元組偏移=0x68C26 → "？？？"
  [ 2] 字元偏移=  7 → 位元組偏移=0x68C2E → "美鈴"
  [ 3] 字元偏移= 10 → 位元組偏移=0x68C34 → "女"
  [ 4] 字元偏移= 12 → 位元組偏移=0x68C38 → "女子学生１"
  [ 5] 字元偏移= 18 → 位元組偏移=0x68C44 → "男子学生１"
  [ 6] 字元偏移= 24 → 位元組偏移=0x68C50 → "男子学生２"
  [ 7] 字元偏移= 30 → 位元組偏移=0x68C5C → "女子学生２"
  [ 8] 字元偏移= 36 → 位元組偏移=0x68C68 → "男子学生３"
  [ 9] 字元偏移= 42 → 位元組偏移=0x68C74 → "男子学生４"
  [10] 字元偏移= 48 → 位元組偏移=0x68C80 → "男子学生５"
  [11] 字元偏移= 54 → 位元組偏移=0x68C8C → "男子学生６"
  [12] 字元偏移= 60 → 位元組偏移=0x68C98 → "男子学生７"
  [13] 字元偏移= 66 → 位元組偏移=0x68CA4 → "リディム"
  [14] 字元偏移= 71 → 位元組偏移=0x68CAE → "男子学生５・男子学生６・男子学生７"
  [15] 字元偏移= 89 → 位元組偏移=0x68CD2 → "男子学生８"
  [16] 字元偏移= 95 → 位元組偏移=0x68CDE → "着物集団"
  [17] 字元偏移=100 → 位元組偏移=0x68CE8 → "雪乃"
  [18] 字元偏移=103 → 位元組偏移=0x68CEE → "衛士"
  [19] 字元偏移=106 → 位元組偏移=0x68CF4 → "学生たち"
  [20] 字元偏移=111 → 位元組偏移=0x68CFE → "シャスティ"


## 6. 解析對話索引表 (Dialog Index Table)

對話索引表位於 `0x68D10`，包含 1344 個對話的字元偏移量。

### 對話索引表結構

```
0x68D10: [索引0] [索引1] [索引2] ... [索引1343]
         (1344 × 4 bytes = 5376 bytes)
```

對話字串表從 `0x6A210` 開始，計算實際位置的公式：

```
實際位元組偏移量 = 對話字串表起始 + (索引值 × 2)
                = 0x6A210 + (index × 2)
```

In [11]:
# 對話索引表和字串表的位置常數
DIALOG_INDEX_TABLE = 0x68D10
DIALOG_STRING_TABLE = 0x6A210
DIALOG_INDEX_COUNT = (DIALOG_STRING_TABLE - DIALOG_INDEX_TABLE) // 4  # 計算索引數量

print("【對話索引表解析】")
print(f"  索引表位置: 0x{DIALOG_INDEX_TABLE:05X}")
print(f"  字串表位置: 0x{DIALOG_STRING_TABLE:05X}")
print(f"  索引數量: {DIALOG_INDEX_COUNT}")
print()

# 讀取前 10 個對話索引作為範例
print("【前 10 個對話索引】")
dialog_indices = []
for i in range(10):
    offset = DIALOG_INDEX_TABLE + i * 4
    char_index = struct.unpack_from('<I', data, offset)[0]
    dialog_indices.append(char_index)
    print(f"  索引 [{i}] 位置 0x{offset:05X}: 字元偏移量 = {char_index}")

【對話索引表解析】
  索引表位置: 0x68D10
  字串表位置: 0x6A210
  索引數量: 1344

【前 10 個對話索引】
  索引 [0] 位置 0x68D10: 字元偏移量 = 0
  索引 [1] 位置 0x68D14: 字元偏移量 = 18
  索引 [2] 位置 0x68D18: 字元偏移量 = 47
  索引 [3] 位置 0x68D1C: 字元偏移量 = 70
  索引 [4] 位置 0x68D20: 字元偏移量 = 103
  索引 [5] 位置 0x68D24: 字元偏移量 = 141
  索引 [6] 位置 0x68D28: 字元偏移量 = 167
  索引 [7] 位置 0x68D2C: 字元偏移量 = 180
  索引 [8] 位置 0x68D30: 字元偏移量 = 212
  索引 [9] 位置 0x68D34: 字元偏移量 = 267


## 7. 提取遊戲文本內容

現在我們已經知道如何定位索引表，接下來就可以提取實際的遊戲文本了。

### 提取流程

1. 讀取索引表中的字元偏移量
2. 計算實際的位元組偏移量
3. 從該位置讀取 UTF-16LE 字串直到遇到 null terminator

In [12]:
# 使用對話索引表讀取對話文本
print("【前 10 個對話文本】\n")

for i, char_index in enumerate(dialog_indices):
    # 計算實際位元組偏移量
    byte_offset = DIALOG_STRING_TABLE + char_index * 2
    
    # 讀取字串
    dialog = read_utf16le_string(data, byte_offset)
    
    # 截斷過長的對話以便顯示
    display_text = dialog[:50] + "..." if len(dialog) > 50 else dialog
    
    print(f"對話 {i:3d}: [偏移=0x{byte_offset:05X}]")
    print(f"          {display_text}")
    print()

【前 10 個對話文本】

對話   0: [偏移=0x6A210]
          「……おい、あの女はどうなった？」

對話   1: [偏移=0x6A234]
          「……はい、お言いつけどおり、捕らえて監禁してあります」

對話   2: [偏移=0x6A26E]
          完全なる闇夜に包まれた、とある部屋の一角……

對話   3: [偏移=0x6A29C]
          俺は近待のメイド、美鈴（メイリン）と密やかな会話を交わしていた。

對話   4: [偏移=0x6A2DE]
          「今は、坊ちゃまの寝室に拘束してあります。そろそろ目覚める頃かと思います」

對話   5: [偏移=0x6A32A]
          「そうか……気が利くな。じゃ、その顔を見に行くか」

對話   6: [偏移=0x6A35E]
          「はい。どうぞ坊ちゃま」

對話   7: [偏移=0x6A378]
          俺の前を率先して歩く美鈴のあとに続き、自分の寝室へ足を進める。

對話   8: [偏移=0x6A3B8]
          「あうぅう……！　んぐっ、うくぅうぅ！　は、放せぇ！　このっ、変態！　こんなことして……どういうつも...

對話   9: [偏移=0x6A426]
          寝室に入ったとたん、なんともはしたない女の罵声が勢いよく耳に飛び込んでくる。



## 8. 文本解碼與顯示

### 完整的文本提取函式

以下是一個完整的函式，可以提取所有遊戲文本：

In [13]:
def extract_all_game_text(data: bytes) -> dict:
    """
    提取所有遊戲文本（名稱和對話）
    
    Args:
        data: 二進位檔案資料
        
    Returns:
        包含 'names' 和 'dialogs' 的字典
    """
    # 常數定義
    NAME_INDEX_TABLE = 0x68BC0
    NAME_STRING_TABLE = 0x68C20
    NAME_COUNT = 21
    
    DIALOG_INDEX_TABLE = 0x68D10
    DIALOG_STRING_TABLE = 0x6A210
    DIALOG_COUNT = (DIALOG_STRING_TABLE - DIALOG_INDEX_TABLE) // 4
    
    result = {
        'names': [],
        'dialogs': []
    }
    
    # 提取角色名稱
    for i in range(NAME_COUNT):
        char_index = struct.unpack_from('<I', data, NAME_INDEX_TABLE + i * 4)[0]
        byte_offset = NAME_STRING_TABLE + char_index * 2
        name = read_utf16le_string(data, byte_offset)
        result['names'].append({
            'index': i,
            'char_offset': char_index,
            'byte_offset': byte_offset,
            'text': name
        })
    
    # 提取對話
    for i in range(DIALOG_COUNT):
        char_index = struct.unpack_from('<I', data, DIALOG_INDEX_TABLE + i * 4)[0]
        byte_offset = DIALOG_STRING_TABLE + char_index * 2
        dialog = read_utf16le_string(data, byte_offset)
        result['dialogs'].append({
            'index': i,
            'char_offset': char_index,
            'byte_offset': byte_offset,
            'text': dialog
        })
    
    return result

# 提取所有文本
game_text = extract_all_game_text(data)

print(f"提取完成！")
print(f"  角色名稱: {len(game_text['names'])} 個")
print(f"  對話文本: {len(game_text['dialogs'])} 個")

提取完成！
  角色名稱: 21 個
  對話文本: 1344 個


In [14]:
# 顯示所有角色名稱
print("【所有角色名稱】\n")
for name_entry in game_text['names']:
    print(f"  {name_entry['index']:2d}. {name_entry['text']}")

【所有角色名稱】

   0. \B
   1. ？？？
   2. 美鈴
   3. 女
   4. 女子学生１
   5. 男子学生１
   6. 男子学生２
   7. 女子学生２
   8. 男子学生３
   9. 男子学生４
  10. 男子学生５
  11. 男子学生６
  12. 男子学生７
  13. リディム
  14. 男子学生５・男子学生６・男子学生７
  15. 男子学生８
  16. 着物集団
  17. 雪乃
  18. 衛士
  19. 学生たち
  20. シャスティ


In [15]:
# 顯示部分對話範例
print("【對話範例 (前20條)】\n")
for dialog_entry in game_text['dialogs'][:20]:
    text = dialog_entry['text']
    display_text = text[:60] + "..." if len(text) > 60 else text
    print(f"  [{dialog_entry['index']:4d}] {display_text}")

【對話範例 (前20條)】

  [   0] 「……おい、あの女はどうなった？」
  [   1] 「……はい、お言いつけどおり、捕らえて監禁してあります」
  [   2] 完全なる闇夜に包まれた、とある部屋の一角……
  [   3] 俺は近待のメイド、美鈴（メイリン）と密やかな会話を交わしていた。
  [   4] 「今は、坊ちゃまの寝室に拘束してあります。そろそろ目覚める頃かと思います」
  [   5] 「そうか……気が利くな。じゃ、その顔を見に行くか」
  [   6] 「はい。どうぞ坊ちゃま」
  [   7] 俺の前を率先して歩く美鈴のあとに続き、自分の寝室へ足を進める。
  [   8] 「あうぅう……！　んぐっ、うくぅうぅ！　は、放せぇ！　このっ、変態！　こんなことして……どういうつもりっ！」
  [   9] 寝室に入ったとたん、なんともはしたない女の罵声が勢いよく耳に飛び込んでくる。
  [  10] こんなヤツが、美鈴とともに俺に付き従っていたのかと思うと、主人として嘆かわしいことこの上ないな。
  [  11] 「このっ……聞いてるの！？　こんな、人を連れ去って、縛って！　監禁するなんてッ……！」
  [  12] 「……暗いな。明かりをつけてくれ」
  [  13] ギャアギャアと口汚く罵ってくる女の言葉を無視し、俺は細めた目を美鈴に向け、壁際のスイッチを顎で差す。
  [  14] 「はい、かしこまりました」
  [  15] 恭しい礼とともに、部屋の明かりがいっせいに点灯し、鋭い光に一瞬目を瞬かせる。
  [  16] そして、声がする方へ目を向け直すと――
  [  17] 「うぅうッ、くぅう……！　よくもっ……よくもこんなことぉ！」
  [  18] いつも使っているベッドの上で、エプロンドレスを身に着けた女が、両手両足を拘束されている姿が目に映った。
  [  19] 「この、変態っ！　人を眠らせて、こんなことするなんてッ……よくもやってくれたわね！　許されることだと思ってるの！？」


## 總結

### BSXScript 3.1 檔案結構一覽

| 區域 | 起始位置 | 結束位置 | 說明 |
|------|----------|----------|------|
| Header | 0x00000 | 0x00100 | Magic、版本、偏移量指標 |
| 代碼區 | 0x00100 | 0x68BC0 | 遊戲指令（不可修改） |
| 名稱索引表 | 0x68BC0 | 0x68C14 | 21 × 4 bytes 字元偏移量 |
| 填充區 | 0x68C14 | 0x68C20 | 零填充 |
| 名稱字串表 | 0x68C20 | 0x68D0E | 21 個 UTF-16LE 名稱 |
| 對話索引表 | 0x68D10 | 0x6A210 | 1344 × 4 bytes 字元偏移量 |
| 對話字串表 | 0x6A210 | EOF | UTF-16LE 對話文本 |

### 關鍵 struct 函式

```python
# 讀取單個值
struct.unpack_from('<I', data, offset)  # 讀取小端序 4-byte 整數

# 讀取多個值
struct.unpack('<II', data[0:8])  # 讀取兩個 4-byte 整數

# 寫入值
struct.pack_into('<I', buffer, offset, value)  # 寫入小端序 4-byte 整數
```

### 學習重點

1. **理解二進位檔案結構**：知道每個區域的位置和作用
2. **使用 struct 模組**：正確解析不同格式的數值
3. **索引與偏移量的關係**：字元偏移量 × 2 = 位元組偏移量
4. **UTF-16LE 編碼**：遊戲文本使用的編碼格式

現在您已經掌握了分析 BSXScript 3.1 二進位檔案的基本技能！

## 附錄：實用工具函式

以下是一些在分析二進位檔案時可能用到的工具函式：

In [19]:
def hexdump(data: bytes, offset: int = 0, length: int = 256, bytes_per_line: int = 16) -> None:
    """
    以十六進位格式顯示二進位資料
    
    Args:
        data: 二進位資料
        offset: 起始偏移量
        length: 顯示長度
        bytes_per_line: 每行顯示的位元組數
    """
    end = min(offset + length, len(data))
    
    for i in range(offset, end, bytes_per_line):
        # 偏移量
        line = f"{i:08X}  "
        
        # 十六進位部分
        hex_part = ""
        ascii_part = ""
        for j in range(bytes_per_line):
            if i + j < end:
                b = data[i + j]
                hex_part += f"{b:02X} "
                # ASCII 部分：可印刷字元顯示原字元，否則顯示點
                ascii_part += chr(b) if 32 <= b < 127 else "."
            else:
                hex_part += "   "
                ascii_part += " "
            
            if j == 7:  # 中間分隔
                hex_part += " "
        
        line += hex_part + " |" + ascii_part + "|"
        print(line)

# 使用範例：顯示名稱索引表區域
print("【名稱索引表區域 Hexdump (0x68BC0 - 0x68C20)】\n")
hexdump(data, 0x68BC0, 0x60)

【名稱索引表區域 Hexdump (0x68BC0 - 0x68C20)】

00068BC0  00 00 00 00 03 00 00 00  07 00 00 00 0A 00 00 00  |................|
00068BD0  0C 00 00 00 12 00 00 00  18 00 00 00 1E 00 00 00  |................|
00068BE0  24 00 00 00 2A 00 00 00  30 00 00 00 36 00 00 00  |$...*...0...6...|
00068BF0  3C 00 00 00 42 00 00 00  47 00 00 00 59 00 00 00  |<...B...G...Y...|
00068C00  5F 00 00 00 64 00 00 00  67 00 00 00 6A 00 00 00  |_...d...g...j...|
00068C10  6F 00 00 00 00 00 00 00  00 00 00 00 00 00 00 00  |o...............|


In [20]:
def analyze_file_structure(data: bytes) -> None:
    """
    自動分析並顯示 BSXScript 3.1 檔案結構
    """
    print("=" * 60)
    print("BSXScript 3.1 檔案結構分析報告")
    print("=" * 60)
    
    # 檢查 Magic
    magic = data[0:16].rstrip(b'\x00').decode('ascii', errors='replace')
    print(f"\n【檔案識別】")
    print(f"  Magic: {magic}")
    print(f"  檔案大小: {len(data):,} bytes")
    
    # Header 資訊
    version = struct.unpack_from('<I', data, 0x10)[0]
    name_str_off = struct.unpack_from('<I', data, 0x90)[0]
    dialog_idx_off = struct.unpack_from('<I', data, 0x98)[0]
    dialog_str_off = struct.unpack_from('<I', data, 0xA0)[0]
    
    print(f"\n【Header 資訊】")
    print(f"  Version: 0x{version:X}")
    print(f"  名稱字串表: 0x{name_str_off:05X}")
    print(f"  對話索引表: 0x{dialog_idx_off:05X}")
    print(f"  對話字串表: 0x{dialog_str_off:05X}")
    
    # 區域大小
    name_index_size = name_str_off - 0x68BC0
    name_str_size = dialog_idx_off - name_str_off
    dialog_idx_size = dialog_str_off - dialog_idx_off
    dialog_str_size = len(data) - dialog_str_off
    
    print(f"\n【區域大小】")
    print(f"  代碼區: {0x68BC0:,} bytes ({0x68BC0 / 1024:.1f} KB)")
    print(f"  名稱索引表: {name_index_size} bytes ({name_index_size // 4} 個索引)")
    print(f"  名稱字串表: {name_str_size} bytes")
    print(f"  對話索引表: {dialog_idx_size} bytes ({dialog_idx_size // 4} 個索引)")
    print(f"  對話字串表: {dialog_str_size:,} bytes ({dialog_str_size / 1024:.1f} KB)")
    
    print("\n" + "=" * 60)

# 執行分析
analyze_file_structure(data)

BSXScript 3.1 檔案結構分析報告

【檔案識別】
  Magic: BSXScript 3.1
  檔案大小: 546,842 bytes

【Header 資訊】
  Version: 0x400
  名稱字串表: 0x68C20
  對話索引表: 0x68D10
  對話字串表: 0x6A210

【區域大小】
  代碼區: 428,992 bytes (418.9 KB)
  名稱索引表: 96 bytes (24 個索引)
  名稱字串表: 240 bytes
  對話索引表: 5376 bytes (1344 個索引)
  對話字串表: 112,138 bytes (109.5 KB)

